In [1]:

import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Jan/Reanalysis/Our_Analysis")
META = ROOT / "02_session_metadata"
RESULTS = ROOT / "04_results"
RESULTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(META / "recovered_24_eligible_regions.csv")

genotypes = ["WT", "Fmr1", "Cntnap2", "Shank3B"]

# Anatomically defined regions for the primary analysis.
region_groups = {
    "DG": ["DG-mo", "DG-sg", "DG-po"],
    "CA1": ["CA1"],
    "CA3": ["CA3"],
    "VISp": ["VISp2/3", "VISp4", "VISp5", "VISp6a", "VISp6b"],
    "MOs": ["MOs2/3", "MOs5", "MOs6a"],
    "CP": ["CP"],
    "SUB": ["SUB"],
    "VL": ["VL"],
    "MRN": ["MRN"],
    "APN": ["APN"],
}

# Parent atlas labels and white-matter structures.
excluded_regions = ["HPF", "STR", "MB", "ccb", "cing", "dhc"]

records = []

for group, regions in region_groups.items():
    subset = df[df["region"].isin(regions)]

    row = {
        "region_group": group,
        "subregions": ", ".join(regions),
        "n_subregions": len(subset)
    }

    for genotype in genotypes:
        animals = subset[f"{genotype}_animals"]
        units = subset[f"{genotype}_units"]
        events = subset[f"{genotype}_complete_event_mice"]

        # Without mouse IDs, the exact union across
        # subregions cannot be calculated.
        row[f"{genotype}_mice_lower_bound"] = int(animals.max())
        row[f"{genotype}_units"] = int(units.sum())
        row[f"{genotype}_event_mice_lower_bound"] = int(events.max())

    row["min_mice_lower_bound"] = min(
        row[f"{g}_mice_lower_bound"] for g in genotypes
    )

    row["min_event_mice_lower_bound"] = min(
        row[f"{g}_event_mice_lower_bound"] for g in genotypes
    )

    records.append(row)

grouped = pd.DataFrame(records).sort_values(
    "min_mice_lower_bound", ascending=False
)

grouped["eligible_all_genotypes"] = (
    grouped["min_mice_lower_bound"] >= 5
)

# Save the new analysis matrix permanently.
grouped.to_csv(
    RESULTS / "anatomical_group_coverage.csv",
    index=False
)

df[df["region"].isin(excluded_regions)].to_csv(
    RESULTS / "excluded_atlas_labels.csv",
    index=False
)

display_cols = (
    ["region_group"]
    + [f"{g}_mice_lower_bound" for g in genotypes]
    + [f"{g}_units" for g in genotypes]
    + ["min_event_mice_lower_bound"]
)

print("ANATOMICAL GROUP COVERAGE")
display(grouped[display_cols])

print("\nELIGIBLE GROUPS:")
print(grouped.loc[
    grouped["eligible_all_genotypes"], "region_group"
].tolist())

print("\nSaved to:", RESULTS / "anatomical_group_coverage.csv")


Mounted at /content/drive
ANATOMICAL GROUP COVERAGE


,region_group,WT_mice_lower_bound,Fmr1_mice_lower_bound,Cntnap2_mice_lower_bound,Shank3B_mice_lower_bound,WT_units,Fmr1_units,Cntnap2_units,Shank3B_units,min_event_mice_lower_bound
0,DG,15,13,13,13,1654,2592,865,1552,12
1,CA1,13,11,11,14,1134,1495,1117,1879,11
2,CA3,11,10,10,9,568,324,771,666,8
4,MOs,12,12,8,7,1096,1519,986,979,6
5,CP,9,10,7,6,822,890,807,1070,5
8,MRN,9,11,6,7,2033,1136,1280,796,6
3,VISp,13,11,5,9,1285,1174,594,1039,5
6,SUB,8,10,6,5,808,1293,681,797,5
7,VL,7,8,5,7,131,154,172,567,5
9,APN,7,10,5,7,622,1000,314,586,5



ELIGIBLE GROUPS:
['DG', 'CA1', 'CA3', 'MOs', 'CP', 'MRN', 'VISp', 'SUB', 'VL', 'APN']

Saved to: /content/drive/MyDrive/Jan/Reanalysis/Our_Analysis/04_results/anatomical_group_coverage.csv


In [2]:

from pathlib import Path

folders = [
    Path("/content/drive/MyDrive/Jan/Reanalysis/Our_Analysis"),
    Path("/content/autism_anatomy_inventory"),
]

target_files = {
    "insertion_region_details.csv",
    "region_genotype_details.csv",
    "region_genotype_matrix.csv",
}

for folder in folders:
    print(f"\nChecking: {folder}")
    if not folder.exists():
        print("Not found")
        continue

    found = [
        p for p in folder.rglob("*.csv")
        if p.name in target_files
    ]

    for p in found:
        print("FOUND:", p)

    if not found:
        print("No detailed inventory files found")



Checking: /content/drive/MyDrive/Jan/Reanalysis/Our_Analysis
No detailed inventory files found

Checking: /content/autism_anatomy_inventory
Not found
